In [ ]:
from pathlib import Path
import json
import platform
import sys

import numpy as np
import pandas as pd
import nltk

from sklearn import __version__ as sklearn_version
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import cross_val_score

In [ ]:
MODEL_SEED = 40
SPLIT_SEEDS = [13, 21, 40, 42, 73]

REVIEW_STRATEGIES = [
    "pair_controlled",
    "random_instance",
    "max_cross_split",
]

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn_version)
print("Model random state:", MODEL_SEED)
print("Reviewer split seeds:", SPLIT_SEEDS)

In [ ]:
def find_project_root(start_path=None):
    current = Path(start_path or Path.cwd()).resolve()

    while True:
        if (
            (current / "data" / "original").is_dir()
            and (current / "data" / "review_splits").is_dir()
        ):
            return current

        if current == current.parent:
            break

        current = current.parent

    raise FileNotFoundError(
        "Could not locate the project root containing "
        "data/original/ and data/review_splits/."
    )

In [ ]:
PROJECT_ROOT = find_project_root()

ORIGINAL_DIR = PROJECT_ROOT / "data" / "original"
LEGACY_REORGANIZED_DIR = PROJECT_ROOT / "data" / "reorganized"
REVIEW_SPLITS_DIR = PROJECT_ROOT / "data" / "review_splits"

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "review_tests"
    / "ensemble"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", PROJECT_ROOT)
print("Original:", ORIGINAL_DIR)
print("Legacy reorganized:", LEGACY_REORGANIZED_DIR)
print("Reviewer splits:", REVIEW_SPLITS_DIR)
print("Results:", RESULTS_DIR)

In [ ]:
nltk.download("stopwords")
stop_words = nltk.corpus.stopwords.words("portuguese")

print("Portuguese stopwords:", len(stop_words))

In [ ]:
def load_data(file_path):
    return pd.read_json(
        file_path,
        lines=True,
    )

In [ ]:
def create_x_y(df):
    x = df["text"]
    y = df["label"]
    return x, y

In [ ]:
def load_split_directory(split_dir):
    split_dir = Path(split_dir)

    paths = {
        "train": split_dir / "train.jsonl",
        "validation": split_dir / "validation.jsonl",
        "test": split_dir / "test.jsonl",
    }

    for split_name, path in paths.items():
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing {split_name} file: {path}"
            )

    df_train = load_data(paths["train"])
    df_val = load_data(paths["validation"])
    df_test = load_data(paths["test"])

    return df_train, df_val, df_test

In [ ]:
def build_leal_baseline():
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        stop_words=stop_words,
    )

    rf_model = RandomForestClassifier(
        n_estimators=100,
        criterion="entropy",
        max_depth=15,
        random_state=40,
    )

    lr_model = LogisticRegression(
        random_state=40,
        max_iter=100,
    )

    svm_model = SVC(
        probability=True,
        random_state=40,
    )

    voting_model = VotingClassifier(
        estimators=[
            ("rf", rf_model),
            ("lr", lr_model),
            ("svm", svm_model),
        ],
        voting="soft",
        n_jobs=30,
    )

    return vectorizer, voting_model

In [ ]:
def validate_input_splits(
    df_train,
    df_val,
    df_test,
    run_name,
):
    expected = {
        "train": (df_train, 3990, {0: 1995, 1: 1995}),
        "validation": (df_val, 570, {0: 285, 1: 285}),
        "test": (df_test, 1140, {0: 570, 1: 570}),
    }

    all_ids = []

    for split_name, (
        split_df,
        expected_size,
        expected_classes,
    ) in expected.items():

        if len(split_df) != expected_size:
            raise ValueError(
                f"{run_name}/{split_name}: "
                f"expected {expected_size} examples, "
                f"found {len(split_df)}."
            )

        observed_classes = (
            split_df["label"]
            .astype(int)
            .value_counts()
            .sort_index()
            .to_dict()
        )

        if observed_classes != expected_classes:
            raise ValueError(
                f"{run_name}/{split_name}: "
                f"unexpected class distribution "
                f"{observed_classes}; "
                f"expected {expected_classes}."
            )

        if split_df["id"].duplicated().any():
            raise ValueError(
                f"{run_name}/{split_name}: duplicated IDs."
            )

        all_ids.extend(
            split_df["id"].astype(str).tolist()
        )

    if len(set(all_ids)) != 5700:
        raise ValueError(
            f"{run_name}: train/validation/test do not "
            "contain exactly 5,700 unique IDs."
        )

In [ ]:
def evaluate_predictions(
    y_true,
    y_pred,
):
    report_dict = classification_report(
        y_true,
        y_pred,
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "precision_macro": float(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "recall_macro": float(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "precision_non_pun": float(
            report_dict["0"]["precision"]
        ),
        "recall_non_pun": float(
            report_dict["0"]["recall"]
        ),
        "f1_non_pun": float(
            report_dict["0"]["f1-score"]
        ),
        "precision_pun": float(
            report_dict["1"]["precision"]
        ),
        "recall_pun": float(
            report_dict["1"]["recall"]
        ),
        "f1_pun": float(
            report_dict["1"]["f1-score"]
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    return metrics, report_dict, cm

In [ ]:
def save_run_outputs(
    output_dir,
    df_test,
    y_pred,
    y_probability,
    metrics,
    report_dict,
    cm,
    metadata,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    predictions = pd.DataFrame(
        {
            "id": df_test["id"].astype(str).values,
            "text": df_test["text"].values,
            "true_label": df_test["label"].astype(int).values,
            "predicted_label": np.asarray(y_pred).astype(int),
            "probability_non_pun": y_probability[:, 0],
            "probability_pun": y_probability[:, 1],
        }
    )

    predictions.to_csv(
        output_dir / "predictions.csv",
        index=False,
        encoding="utf-8",
    )

    pd.DataFrame(report_dict).T.to_csv(
        output_dir / "classification_report.csv",
        encoding="utf-8",
    )

    cm_df = pd.DataFrame(
        cm,
        index=["true_non_pun", "true_pun"],
        columns=["pred_non_pun", "pred_pun"],
    )

    cm_df.to_csv(
        output_dir / "confusion_matrix.csv",
        encoding="utf-8",
    )

    with (
        output_dir / "metrics.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metrics,
            file,
            indent=2,
            ensure_ascii=False,
        )

    with (
        output_dir / "metadata.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metadata,
            file,
            indent=2,
            ensure_ascii=False,
        )

In [ ]:
def run_leal_baseline(
    split_dir,
    condition,
    split_seed=None,
    output_dir=None,
):
    split_dir = Path(split_dir)

    print("=" * 90)
    print("Condition:", condition)
    print("Split seed:", split_seed)
    print("Input:", split_dir)

    df_train, df_val, df_test = (
        load_split_directory(
            split_dir
        )
    )

    validate_input_splits(
        df_train=df_train,
        df_val=df_val,
        df_test=df_test,
        run_name=condition,
    )

    x_train, y_train = create_x_y(
        df_train
    )
    x_val, y_val = create_x_y(
        df_val
    )
    x_test, y_test = create_x_y(
        df_test
    )

    vectorizer, voting_model = (
        build_leal_baseline()
    )

    x_train_vectorized = (
        vectorizer.fit_transform(
            x_train
        )
    )

    x_val_vectorized = (
        vectorizer.transform(
            x_val
        )
    )

    x_test_vectorized = (
        vectorizer.transform(
            x_test
        )
    )

    _ = (x_val_vectorized, y_val)

    voting_model.fit(
        x_train_vectorized,
        y_train,
    )

    y_test_pred = voting_model.predict(
        x_test_vectorized
    )

    y_test_probability = (
        voting_model.predict_proba(
            x_test_vectorized
        )
    )

    metrics, report_dict, cm = (
        evaluate_predictions(
            y_true=y_test,
            y_pred=y_test_pred,
        )
    )

    print(
        classification_report(
            y_test,
            y_test_pred,
            zero_division=0,
        )
    )

    print(
        "Accuracy:",
        f"{metrics['accuracy']:.6f}",
    )
    print(
        "Macro-F1:",
        f"{metrics['f1_macro']:.6f}",
    )

    run_result = {
        "condition": condition,
        "split_seed": (
            None
            if split_seed is None
            else int(split_seed)
        ),
        "model_seed": MODEL_SEED,
        **metrics,
    }

    if output_dir is not None:
        metadata = {
            "condition": condition,
            "split_seed": (
                None
                if split_seed is None
                else int(split_seed)
            ),
            "model_seed": MODEL_SEED,
            "input_directory": str(
                split_dir.resolve()
            ),
            "train_examples": len(df_train),
            "validation_examples": len(df_val),
            "test_examples": len(df_test),
            "tfidf_ngram_range": [1, 2],
            "portuguese_stopwords": True,
            "random_forest": {
                "n_estimators": 100,
                "criterion": "entropy",
                "max_depth": 15,
                "random_state": 40,
            },
            "logistic_regression": {
                "random_state": 40,
                "max_iter": 100,
            },
            "svm": {
                "probability": True,
                "random_state": 40,
            },
            "voting": "soft",
            "n_jobs": 30,
            "python": sys.version.split()[0],
            "pandas": pd.__version__,
            "numpy": np.__version__,
            "scikit_learn": sklearn_version,
        }

        save_run_outputs(
            output_dir=output_dir,
            df_test=df_test,
            y_pred=y_test_pred,
            y_probability=y_test_probability,
            metrics=metrics,
            report_dict=report_dict,
            cm=cm,
            metadata=metadata,
        )

    return run_result

In [ ]:
original_result = run_leal_baseline(
    split_dir=ORIGINAL_DIR,
    condition="original",
    split_seed=None,
    output_dir=(
        RESULTS_DIR
        / "original"
    ),
)

pd.DataFrame(
    [original_result]
)

In [ ]:
legacy_reorganized_result = (
    run_leal_baseline(
        split_dir=LEGACY_REORGANIZED_DIR,
        condition="legacy_reorganized",
        split_seed=42,
        output_dir=(
            RESULTS_DIR
            / "legacy_reorganized"
        ),
    )
)

pd.DataFrame(
    [legacy_reorganized_result]
)

In [ ]:
review_results = []

for strategy in REVIEW_STRATEGIES:
    for split_seed in SPLIT_SEEDS:
        split_dir = (
            REVIEW_SPLITS_DIR
            / strategy
            / f"seed_{split_seed}"
        )

        output_dir = (
            RESULTS_DIR
            / strategy
            / f"seed_{split_seed}"
        )

        result = run_leal_baseline(
            split_dir=split_dir,
            condition=strategy,
            split_seed=split_seed,
            output_dir=output_dir,
        )

        review_results.append(
            result
        )

In [ ]:
review_results_df = pd.DataFrame(
    review_results
)

display(review_results_df)

review_results_path = (
    RESULTS_DIR
    / "review_runs.csv"
)

review_results_df.to_csv(
    review_results_path,
    index=False,
    encoding="utf-8",
)

print("Saved:", review_results_path)

In [ ]:
SUMMARY_METRICS = [
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "precision_non_pun",
    "recall_non_pun",
    "f1_non_pun",
    "precision_pun",
    "recall_pun",
    "f1_pun",
]

summary_rows = []

for strategy in REVIEW_STRATEGIES:
    strategy_df = review_results_df[
        review_results_df["condition"]
        == strategy
    ]

    row = {
        "condition": strategy,
        "runs": len(strategy_df),
    }

    for metric in SUMMARY_METRICS:
        row[
            f"{metric}_mean"
        ] = strategy_df[
            metric
        ].mean()

        row[
            f"{metric}_std"
        ] = strategy_df[
            metric
        ].std(
            ddof=1
        )

    summary_rows.append(row)

review_summary_df = pd.DataFrame(
    summary_rows
)

display(review_summary_df)

review_summary_path = (
    RESULTS_DIR
    / "review_summary_mean_std.csv"
)

review_summary_df.to_csv(
    review_summary_path,
    index=False,
    encoding="utf-8",
)

print("Saved:", review_summary_path)

In [ ]:
legacy_results_df = pd.DataFrame(
    [
        original_result,
        legacy_reorganized_result,
    ]
)

legacy_results_path = (
    RESULTS_DIR
    / "legacy_reproduction.csv"
)

legacy_results_df.to_csv(
    legacy_results_path,
    index=False,
    encoding="utf-8",
)

display(legacy_results_df)

print("Saved:", legacy_results_path)

In [ ]:
compact_rows = [
    {
        "condition": "original",
        "accuracy": (
            f"{original_result['accuracy']:.4f}"
        ),
        "macro_f1": (
            f"{original_result['f1_macro']:.4f}"
        ),
        "type": "fixed reference",
    },
    {
        "condition": "legacy_reorganized",
        "accuracy": (
            f"{legacy_reorganized_result['accuracy']:.4f}"
        ),
        "macro_f1": (
            f"{legacy_reorganized_result['f1_macro']:.4f}"
        ),
        "type": "fixed legacy reference",
    },
]

for _, row in review_summary_df.iterrows():
    compact_rows.append(
        {
            "condition": row["condition"],
            "accuracy": (
                f"{row['accuracy_mean']:.4f} "
                f"± {row['accuracy_std']:.4f}"
            ),
            "macro_f1": (
                f"{row['f1_macro_mean']:.4f} "
                f"± {row['f1_macro_std']:.4f}"
            ),
            "type": "5 split seeds",
        }
    )

compact_comparison_df = pd.DataFrame(
    compact_rows
)

display(compact_comparison_df)

compact_path = (
    RESULTS_DIR
    / "compact_comparison.csv"
)

compact_comparison_df.to_csv(
    compact_path,
    index=False,
    encoding="utf-8",
)

print("Saved:", compact_path)

In [ ]:
RUN_LEAL_CV_REPRODUCTION = False

if RUN_LEAL_CV_REPRODUCTION:
    df_train, df_val, df_test = (
        load_split_directory(
            ORIGINAL_DIR
        )
    )

    data = pd.concat(
        [
            df_train,
            df_val,
            df_test,
        ]
    )

    X, y = create_x_y(data)

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        stop_words=stop_words,
    )

    x_all_vectorized = (
        vectorizer.fit_transform(X)
    )

    _, voting_model = (
        build_leal_baseline()
    )

    scores = cross_val_score(
        voting_model,
        x_all_vectorized,
        y,
        cv=5,
    )

    print(
        "Fold scores:",
        scores,
    )
    print(
        "Mean:",
        scores.mean(),
    )
    print(
        "Std:",
        scores.std(ddof=1),
    )
else:
    print(
        "Leal et al. CV reproduction skipped. "
        "Set RUN_LEAL_CV_REPRODUCTION = True "
        "only if you want to reproduce that legacy check."
    )